# University research output indicators

Run all cells to reproduce the historical and institutional analyses and download the results. No file upload is required.

In [ ]:
from pathlib import Path
import json, subprocess, sys, tempfile
from importlib import metadata

source = "https://github.com/evidenceworks/university-portfolio-validity.git"
workspace = Path(tempfile.mkdtemp(prefix="university-output-"))
repository = workspace / "repository"
subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", "main", source, str(repository)], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repository, text=True).strip()
print("Public source:", source.removesuffix(".git"))
print("Commit:", commit)

def reproduce(mode):
    run = subprocess.run([sys.executable, "reproduce.py", mode], cwd=repository, text=True, capture_output=True)
    if run.returncode != 0:
        print(run.stdout, run.stderr)
        raise RuntimeError(f"{mode.capitalize()} reproduction failed")
    report = json.loads(run.stdout[run.stdout.index("{"):])
    assert report["status"] == mode.upper() + "_PASS"
    assert report["network"] == "blocked"
    assert report["verified_payload_files"] == 17
    return report


In [ ]:
quick = reproduce("quick")
assert quick["historical_decisions_reproduced"] == 6400
assert quick["institutional_decisions_reproduced"] == 71264
print("QUICK_PASS | 17/17 SHA-256 | 6,400 historical and 71,264 institutional decisions")


In [ ]:
try:
    numpy_version = metadata.version("numpy")
except metadata.PackageNotFoundError:
    numpy_version = None
if numpy_version != "2.3.5":
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "-r", str(repository / "requirements.txt")], check=True)
core = reproduce("core")
assert (core["numerical_tables_checked"], core["reference_rows_checked"], core["reference_cells_checked"]) == (15, 940, 7511)
assert core["full_frame_selection"]["historical_selections_verified"] == 400
assert core["full_frame_selection"]["institutional_selections_verified"] == 2000
coverage = core["exact_coverage_checks"]
assert coverage["finite_population_one_sided_coverage_checks"] == 12312
assert coverage["two_sided_coverage_checks"] == 3078
print("CORE_PASS | 15 numerical tables | 940 reference rows | 7,511 values")
print("Sample selection: 400 historical pairs and 2,000 institutional pairs")
print("Exact coverage: 12,312 one-sided and 3,078 two-sided checks")


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

result_files = sorted((repository / "results").iterdir())
assert len(result_files) == 19 and all(path.is_file() for path in result_files)
archive = (Path("/content") if Path("/content").is_dir() else workspace) / "university_output_results.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in result_files:
        bundle.write(path, path.name)
with ZipFile(archive) as bundle:
    assert bundle.testzip() is None and len(bundle.namelist()) == 19
print("Results archive:", archive.name, "| 19 files")
try:
    from google.colab import files
except ImportError:
    print("Saved:", archive)
else:
    files.download(str(archive))
